In [ ]:
import os
import pandas as pd
import swifter 
import numpy as np
import string
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

nltk_path = os.path.join(os.path.dirname(nltk.__file__), 'nltk_data')
nltk.data.path.append(nltk_path)
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

force_prep = False # set to True if we want to preprocess data by force

if os.path.exists('prepped_polusa.csv') and not force_prep:
    prepped_polusa = pd.read_csv('prepped_polusa.csv')
    X_train, X_test, y_train, y_test = train_test_split(prepped_polusa['text'], prepped_polusa['label'], test_size=0.2, random_state=42)
else:
    # Combine all POLUSA dataset
    files = [
            # 'polusa/2017_1.csv', 'polusa/2017_2.csv', 
            # 'polusa/2018_1.csv', 'polusa/2018_2.csv',
            'polusa/2019_1.csv', 'polusa/2019_2.csv']

    polusa = [pd.read_csv(f) for f in files]
    combined_polusa = pd.concat(polusa, ignore_index=True)

    # Drop undefined labels
    combined_polusa = combined_polusa[combined_polusa['political_leaning'] != 'UNDEFINED']

    # Drop non-predictive columns
    cd_polusa = combined_polusa.drop(['id', 'url', 'date_publish', 'authors', 'domain'], axis=1)

    # Combine article into one single text
    cd_polusa['text'] = (cd_polusa['outlet'].fillna('') + ' ' + 
                        cd_polusa['headline'].fillna('') + ' ' + 
                        cd_polusa['lead'].fillna('') + ' ' + 
                        cd_polusa['body'].fillna(''))

    # Convert political leaning into numbers
    # Center = 0, Left = 1, Right = 2
    le = LabelEncoder()
    cd_polusa['label'] = le.fit_transform(cd_polusa['political_leaning'])
    
    cleaned_polusa = cd_polusa[['text','label']]
    cleaned_polusa.head()

    X_train, X_test, y_train, y_test = train_test_split(cleaned_polusa['text'], cleaned_polusa['label'], test_size=0.2, random_state=42)

    stop_words = set(stopwords.words('english'))
    def preprocess(txt):
        txt = txt.lower()
        txt = ''.join([c for c in txt if c not in string.punctuation])
        toks = word_tokenize(txt)
        toks = [word for word in toks if word not in stop_words]
        return ' '.join(toks)

    X_train = X_train.swifter.apply(preprocess)
    X_test = X_test.swifter.apply(preprocess)

    prepped_polusa = pd.DataFrame({'text': pd.concat([X_train, X_test]), 'label': pd.concat([y_train, y_test])})
    prepped_polusa.to_csv('prepped_polusa.csv', index=False)
    prepped_polusa.head()

X_train
X_test



In [ ]:
print(prepped_polusa.columns)
print(prepped_polusa.dtypes)

print(prepped_polusa['label'].value_counts())

In [ ]:
from keras._tf_keras.keras.preprocessing.text import Tokenizer
from keras._tf_keras.keras.preprocessing.sequence import pad_sequences

tokenizer = Tokenizer()
tokenizer.fit_on_texts(X_train)
X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

max_length = 100
X_train = pad_sequences(X_train, maxlen=max_length, padding='post')
X_test = pad_sequences(X_test, maxlen=max_length, padding='post')

In [ ]:
# Load GloVe
embeddings_index = {}
with open('glove/glove.6B.100d.txt', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        coefs = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = coefs

print(f'Found {len(embeddings_index)} word vectors.')


In [ ]:
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

# def find_root(word, glove_embeddings, min_len=3):
#     substrings = [word[i:j] for i in range(len(word)) for j in range(i+min_len, len(word)+1)]
#     substrings.sort(key=len, reverse=True)  # longest first
#     for sub in substrings:
#         if sub in glove_embeddings:
#             return sub
#     return None

# def estimate_embedding(word, glove_embeddings):
#     root = find_root(word, glove_embeddings)
#     if root:
#         return glove_embeddings[root]
#     else:
#         return None

In [ ]:
def find_roots(word, glove_embeddings, min_len=3, top_k=2):
    """
    Find top_k longest, non-overlapping root words in the OOV word.
    """
    # Find all candidate roots
    candidates = []
    for i in range(len(word)):
        for j in range(i + min_len, len(word) + 1):
            sub = word[i:j]
            if sub in glove_embeddings:
                candidates.append((sub, i, j))  # store substring and its indices

    # Sort candidates by length descending
    candidates.sort(key=lambda x: len(x[0]), reverse=True)

    selected = []
    occupied = set()  # track indices already covered
    for sub, start, end in candidates:
        if not any(idx in occupied for idx in range(start, end)):
            selected.append(sub)
            occupied.update(range(start, end))
        if len(selected) >= top_k:
            break

    return selected

def estimate_embedding(word, glove_embeddings, min_len=3, top_k=2):
    """
    Estimate OOV embedding by aggregating embeddings of top_k
    longest non-overlapping root words (weighted by length).
    """
    # Exact match
    if word in glove_embeddings:
        return glove_embeddings[word]

    roots = find_roots(word, glove_embeddings, min_len=min_len, top_k=top_k)
    if roots:
        vecs = np.array([glove_embeddings[r] for r in roots])
        weights = np.array([len(r) for r in roots], dtype=np.float32)
        weights /= np.sum(weights)  # normalize
        return np.sum(vecs * weights[:, np.newaxis], axis=0)

In [ ]:
# Prepare embedding matrix
embedding_dim = 100
word_index = tokenizer.word_index
num_words = len(word_index) + 1
embedding_matrix = np.zeros((num_words, embedding_dim))

for word, i in word_index.items():
    embedding_vector = embeddings_index.get(word)
    if embedding_vector is not None:
        embedding_matrix[i] = embedding_vector
    else:
        estimated_vec = estimate_embedding(word, embeddings_index)
        if estimated_vec is not None:
            embedding_matrix[i] = estimated_vec

In [ ]:
import tensorflow as tf
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
        tf.config.experimental.set_visible_devices(gpus[0], 'GPU')
        print('Using GPU:', gpus[0])
    except RuntimeError as e:
        print(e)
else:
    print('GPU Not found')


In [ ]:
from keras._tf_keras.keras.models import Sequential
from keras._tf_keras.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from keras._tf_keras.keras.optimizers import Adam
from keras._tf_keras.keras.regularizers import l2

# Clear previous models and free memory
import gc
tf.keras.backend.clear_session()
gc.collect()

model = Sequential()

# GloVe-based Embedding layer
model.add(Embedding(
    input_dim=num_words,
    output_dim=embedding_dim,
    weights=[embedding_matrix],
    trainable=False   # keep GloVe fixed initially
))

# First Bidirectional BiLSTM layer 
model.add(Bidirectional(LSTM(64, dropout=0.3, return_sequences=True)))

# Second Bidirectional BiLSTM layer 
model.add(Bidirectional(LSTM(64, dropout=0.3)))

# Fully connected layers
model.add(Dense(64, activation='relu', kernel_regularizer=l2(0.01)))
model.add(Dropout(0.3))

# Template for 1-layer BiLSTM
# model.add(Bidirectional(LSTM(32, dropout=0.1)))
# model.add(Dense(32, activation='relu', kernel_regularizer=l2(0.01)))
# model.add(Dropout(0.1))

# Output layer
model.add(Dense(3, activation='softmax'))


In [ ]:
from keras._tf_keras.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss',
                           patience=3,  # how many epochs with no improvement before stopping
                           restore_best_weights=True)

In [ ]:
import keras._tf_keras.keras.backend as keras_backend

# Class weights for prepped_polusa.csv
class_weights = {
    0: 1.014,
    1: 0.769,
    2: 1.398
}

# Class weights for prepped_polusa2.csv (larger dataset)
# class_weights = {
#     0: 0.956, 
#     1: 0.789,  
#     2: 1.457   
# }

gc.collect()  
model.compile(optimizer=Adam(learning_rate=0.001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
history = model.fit(np.array(X_train), np.array(y_train), epochs=35, batch_size=356, class_weight=class_weights, validation_data=(X_test, y_test), callbacks=[early_stop])
gc.collect()


In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay
import matplotlib

y_pred = model.predict(X_test)
y_pred_classes = np.argmax(y_pred, axis=1)

# Compute evaluation metrics
print(f'Accuracy: {accuracy_score(y_test, y_pred_classes):.4f}')
print(f'Precision: {precision_score(y_test, y_pred_classes, average='weighted'):.4f}')
print(f'Recall: {recall_score(y_test, y_pred_classes, average='weighted'):.4f}')
print(f'F1 Score: {f1_score(y_test, y_pred_classes, average='weighted'):.4f}')

# Show confusion matrix
print('\nConfusion Matrix:')
conf_matrix = confusion_matrix(y_test, y_pred_classes)
print(conf_matrix)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix, display_labels=['Center', 'Left', 'Right'])
disp.plot()
gc.collect()


In [ ]:
print(model.summary())

In [ ]:
import matplotlib.pyplot as plt

plt.plot(history.history['loss'], label='train_loss')
plt.plot(history.history['val_loss'], label='val_loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.show()

plt.plot(history.history['accuracy'], label='train_acc')
plt.plot(history.history['val_accuracy'], label='val_acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()


In [ ]:
# Save the full model
model.save('polusa_BiLSTM_v2.keras')